# Setup workflow environment

In [6]:
using Pkg; Pkg.activate(dirname(Base.current_project()))

import TulipaIO as TIO
using DuckDB
using DataFrames
using Plots

# For Win. system to fix the KaTex parse error in Jupyter Notebook
Base.show(stdout, ::MIME"text/latex", df::DataFrame) = show(stdout, MIME("text/plain"), df)

#= For utility functions
    - print_annual_total_prod(DBconnection, years...)
    - inv_cost, fom_cost, var_cost = objective_terms_value(TulipaProblem, DBconnection)
    - annual_flows_between_assets(DB_conn, from_asset_term, to_asset_term)
=#
include("../src/util_analysis.jl")

# For saved figures: '.pdf' for Latex, '.svg' for slides.
fig_format = "pdf";

  Activating project at `c:\ModellingRepos\VintageDemo`


# Data preparation

- Rebuild DB connection of output and input data such that the plotting can be independent of model runs.
- Plotting code [reference](https://docs.juliaplots.org/dev/gallery/gr/generated/gr-ref056/)

In [7]:
!(@isdefined connection_no_vintage) && begin 
    connection_no_vintage = DBInterface.connect(DuckDB.DB) 
    TIO.read_csv_folder(connection_no_vintage, "model-instance-Tulipa/input-1-no-vintage")
    TIO.read_csv_folder(connection_no_vintage, "model-instance-Tulipa/output-1-no-vintage")
end

!(@isdefined connection_vintage_standard) && begin 
    connection_vintage_standard = DBInterface.connect(DuckDB.DB) 
    TIO.read_csv_folder(connection_vintage_standard, "model-instance-Tulipa/input-2-vintage-standard")
    TIO.read_csv_folder(connection_vintage_standard, "model-instance-Tulipa/output-2-vintage-standard")
end

!(@isdefined connection_vintage_compact) && begin 
    connection_vintage_compact = DBInterface.connect(DuckDB.DB) 
    TIO.read_csv_folder(connection_vintage_compact, "model-instance-Tulipa/input-3-vintage-compact")
    TIO.read_csv_folder(connection_vintage_compact, "model-instance-Tulipa/output-3-vintage-compact")
end;

## 1. Wind installation over years

In [8]:
assets_investment_no_vintage = filter(row -> row.asset == "wind", TIO.get_table(connection_no_vintage, "var_assets_investment"))
assets_investment_vintage_standard = filter(row -> occursin("wind", row.asset), TIO.get_table(connection_vintage_standard, "var_assets_investment"))
assets_investment_vintage_compact = filter(row -> row.asset == "wind", TIO.get_table(connection_vintage_compact, "var_assets_investment"))

fig_cap = plot(
    (assets_investment_no_vintage.milestone_year .- 1), assets_investment_no_vintage.solution,
    label="No vintage",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    # markersize=7, markerstrokewidth=3,
    # markershape=:xcross,
    color=:green, alpha = 0.45,
    xlabel="Milestone year", xguidefontsize=12, 
    ylabel="Offshore wind capacity (GW)\nannual addition", yguidefontsize=12,
    yticks = 0:20:120, ylims = (0, maximum(assets_investment_no_vintage.solution) * 1.1),
    xticks=minimum(assets_investment_no_vintage.milestone_year):10:maximum(assets_investment_no_vintage.milestone_year),
    xtickfontsize=12,ytickfontsize=12,
    legend=:topright, legendfontsize=11,
    legend_column = 1,
)
plot!(
    fig_cap, assets_investment_vintage_standard.milestone_year, assets_investment_vintage_standard.solution,
    label="Vintage standard",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    # markersize=7, markerstrokewidth=3,
    # markershape=:circle,
    color=:red, alpha = 0.45,
)
plot!(
    fig_cap, (assets_investment_vintage_compact.milestone_year .+ 1), assets_investment_vintage_compact.solution,
    label="Vintage compact",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    # markersize=9, markerstrokewidth=3
    # markershape=:cross,
    color=:blue, alpha = 0.45,
)

fig_cap

savefig(fig_cap, joinpath(@__DIR__, "result_plots", "wind_investment_capacity.$(fig_format)"));

## 2. Power generation over years

In [9]:
wind_to_demand_no_vintage = annual_flows_between_assets(connection_no_vintage, "wind", "offshorehub")
wind_to_demand_vintage_standard = annual_flows_between_assets(connection_vintage_standard, "wind", "offshorehub")
wind_to_demand_vintage_compact = annual_flows_between_assets(connection_vintage_compact, "wind", "offshorehub")

ens_to_demand_no_vintage = annual_flows_between_assets(connection_no_vintage, "ens", "demand")
ens_to_demand_vintage_standard = annual_flows_between_assets(connection_vintage_standard, "ens", "demand")
ens_to_demand_vintage_compact = annual_flows_between_assets(connection_vintage_compact, "ens", "demand")

fig_prod = plot(
    (wind_to_demand_no_vintage.milestone_year .- 1), wind_to_demand_no_vintage.annual_flow,
    label="Wind production - No vintage",
    seriestype=:bar,
    bar_edge = :false,
    # bar_position = :overlay,
    bar_width = 1,
    color=:green, alpha = 0.45,
    yticks = 0:300:1500, ylims = (0, maximum(wind_to_demand_no_vintage.annual_flow + ens_to_demand_no_vintage.annual_flow) * 1.1),
    xticks=minimum(wind_to_demand_no_vintage.milestone_year):10:maximum(wind_to_demand_no_vintage.milestone_year),
    xtickfontsize=12,ytickfontsize=12,
    xlabel="Milestone year", xguidefontsize=12, 
    ylabel="Electrical energy (TWh)", yguidefontsize=12,
    legend=:bottomright, legendfontsize=11,
    legend_column = 1,
)
plot!(
    fig_prod, wind_to_demand_vintage_standard.milestone_year, wind_to_demand_vintage_standard.annual_flow,
    label="Wind production - Vintage standard",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    color=:red, alpha = 0.45,
)
plot!(
    fig_prod, (wind_to_demand_vintage_compact.milestone_year .+ 1), wind_to_demand_vintage_compact.annual_flow,
    label="Wind production - Vintage compact",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    color=:blue, alpha = 0.45,
)

plot!(
    fig_prod, (ens_to_demand_no_vintage.milestone_year .- 1), 
    ens_to_demand_no_vintage.annual_flow .+ wind_to_demand_no_vintage.annual_flow,
    fillto = wind_to_demand_no_vintage.annual_flow,
    label="Market purchase - All methods",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    color=:black, alpha = 1,
    fillstyle = :/,
)
plot!(
    fig_prod, ens_to_demand_vintage_standard.milestone_year, 
    ens_to_demand_vintage_standard.annual_flow .+ wind_to_demand_vintage_standard.annual_flow,
    fillto = wind_to_demand_vintage_standard.annual_flow,
    label="",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    color=:black, alpha = 1,
    fillstyle = :/,
)
plot!(
    fig_prod, (ens_to_demand_vintage_compact.milestone_year .+ 1), 
    ens_to_demand_vintage_compact.annual_flow .+ wind_to_demand_vintage_compact.annual_flow,
    fillto = wind_to_demand_vintage_compact.annual_flow,
    label="",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 1,
    color=:black, alpha = 1,
    fillstyle = :/,
)

fig_prod

savefig(fig_prod, joinpath(@__DIR__, "result_plots", "annual_productions.$(fig_format)"));

## 3. Total system costs

In [10]:
instances = ["No vintage", "Vintage standard", "Vintage compact"]
x_location = 1:3:length(instances)*3

system_costs_no_vintage = total_system_costs_category(connection_no_vintage)
system_costs_vintage_standard = total_system_costs_category(connection_vintage_standard)
system_costs_vintage_compact = total_system_costs_category(connection_vintage_compact)

inv_costs = [system_costs_no_vintage.inv_cost, system_costs_vintage_standard.inv_cost, system_costs_vintage_compact.inv_cost]/1000
fom_costs = [system_costs_no_vintage.fixed_om_cost, system_costs_vintage_standard.fixed_om_cost, system_costs_vintage_compact.fixed_om_cost]/1000
var_costs = [system_costs_no_vintage.variable_om_cost, system_costs_vintage_standard.variable_om_cost, system_costs_vintage_compact.variable_om_cost]/1000
total_costs = [system_costs_no_vintage.total_cost, system_costs_vintage_standard.total_cost, system_costs_vintage_compact.total_cost]/1000

fig_costs = plot(
    x_location, inv_costs,
    label="Investments",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 2,
    color=:green, alpha = 0.45,
    yticks = 0:200:1000, ylims = (0, maximum(total_costs) * 1.1),
    # top_margin = 6Plots.mm,
    xticks = (x_location, instances),
    xtickfontsize=12,ytickfontsize=12,
    # xlabel="Model instance", xguidefontsize=12, 
    ylabel="Total system cost (billion €)", yguidefontsize=12,
    legend=:outertop , legendfontsize=11,
    legend_column = -1
)
plot!(
    fig_costs, x_location, (fom_costs .+ inv_costs),
    fillto = inv_costs,
    label="Fixed O&M",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 2,
    color=:red, alpha = 0.45,
)
plot!(
    fig_costs, x_location, (var_costs .+ fom_costs .+ inv_costs),
    fillto = (fom_costs .+ inv_costs),
    label="Variable O&M",
    seriestype=:bar,
    bar_edge = :false,
    bar_width = 2,
    color=:blue, alpha = 0.45,
)

# add segment labels
for i in eachindex(x_location)
    annotate!(fig_costs, x_location[i], inv_costs[i]/2, text("$(round(inv_costs[i], digits=1))", 12, :center, :center))
    annotate!(fig_costs, x_location[i], inv_costs[i] + fom_costs[i]/2, text("$(round(fom_costs[i], digits=1))", 12, :center, :center))
    annotate!(fig_costs, x_location[i], inv_costs[i] + fom_costs[i] + var_costs[i]/2, text("$(round(var_costs[i], digits=1))", 12, :center, :center))
    annotate!(fig_costs, x_location[i], total_costs[i] + 10, text("Sum: $(round(total_costs[i], digits=1))", 12, :center, :bottom))
end

fig_costs

savefig(fig_costs, joinpath(@__DIR__, "result_plots", "total_costs.$(fig_format)"));